# Notebook 안내
원본 팀 연구 코드의 계산 로직은 변경하지 않았다. 가독성을 위한 설명 셀만 추가했으며 개인정보·데이터 노출 방지를 위해 기존 실행 출력은 제거했다. 원천 데이터는 포함하지 않으며 독립 재실행은 하지 않았다. README의 결과 및 한계 참고.


# Earnings Surprise × EPS Revision — 3M Lag + Liquidity Filter Long-Short

기존 Long-Short 전략에서 **FY1을 제거하고 FY2 EPS Revision만 사용**하도록 수정한 버전이다.

- 유니버스: 매월 **시가총액 3,000억원 이상**
- EPS Revision: **FY2만 사용**
- Signal availability lag: **3개월**
- Earnings Surprise와 EPS Revision 모두 formation month 기준 **3개월 전 신호** 사용
- Liquidity filter: **60거래일 평균 거래대금** 기준 횡단면 **하위 20% 제외**
- Combined Score: `50% EPS Revision Rank + 50% Earnings Surprise Rank`
- Long: Combined Score 상위 20개, **+100%**
- Short: Combined Score 하위 20개, **-100%**
- 정상 운용월: **Net Exposure = 0%, Gross Exposure = 200%**
- 월말 formation → 다음 달 holding
- 거래비용: 왕복 30bp = 편도 15bp
- 대차비용: 연 3%
- 전면 공매도 금지와 겹치는 월은 포지션을 0으로 두어 전략 운용에서 제외

신호 시점은 다음과 같다.

$$
Signal_{t-3}\rightarrow Formation_t\rightarrow Holding_{t+1}
$$

따라서 `SIGNAL_LAG_MONTHS = 3`은 **신호 availability lag**이고, 기존의 다음 달 보유 구조는 별도로 유지한다.


### 코드 설명 — 환경 설정: pandas·NumPy·statsmodels 및 전략 파라미터를 정의한다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display

BACKTEST_START = pd.Timestamp('2015-10-31')

N_LONG = 20
N_SHORT = 20
MCAP_MIN_EOK = 3000
SIGNAL_LAG_MONTHS = 3
TRADING_VALUE_LOOKBACK = 60
LIQUIDITY_BOTTOM_EXCLUDE = 0.20

SURPRISE_WEIGHT = 0.50
REVISION_WEIGHT = 0.50

ROUND_TRIP_COST_BPS = 30.0
ONE_WAY_COST = ROUND_TRIP_COST_BPS / 2 / 10_000

ANNUAL_BORROW_COST = 0.03
MONTHLY_BORROW_COST = (1 + ANNUAL_BORROW_COST) ** (1 / 12) - 1

LIQUIDATE_AT_END = True

# 월 단위 백테스트이므로, 아래 전면 공매도 금지기간과 하루라도 겹치는 월은 보수적으로 전략 미운용
SHORT_BAN_RANGES = [
    (pd.Timestamp('2020-03-16'), pd.Timestamp('2021-05-02')),
    (pd.Timestamp('2023-11-06'), pd.Timestamp('2025-03-30')),
]

print(f'Signal lag = {SIGNAL_LAG_MONTHS} months')
print('Revision Score = FY2 only')
print(f'Liquidity filter = bottom {LIQUIDITY_BOTTOM_EXCLUDE:.0%} excluded by {TRADING_VALUE_LOOKBACK}-day avg trading value')
print(f'Long / Short = +100% / -100%, Gross = 200%, Net = 0% in active months')
print(f'Transaction cost = {ROUND_TRIP_COST_BPS:.0f}bp round trip ({ONE_WAY_COST*10_000:.0f}bp per side)')
print(f'Borrow cost = {ANNUAL_BORROW_COST:.1%} annual = {MONTHLY_BORROW_COST:.4%} monthly')


## 1. 파일 경로


### 코드 설명 — 입력 파일 탐색: 경로 후보를 검사한다. 실제 ZIP의 Date 폴더명과 코드의 Data 폴더명이 다르므로 실행 전 확인한다.


In [ ]:
def resolve_file(candidates, label, required=True):
    for p in candidates:
        if Path(p).exists():
            print(f'{label}: {p}')
            return p
    if required:
        raise FileNotFoundError(
            f'{label} not found. Tried:\n' + '\n'.join(candidates)
        )
    print(f'{label}: not found')
    return None

FILES = {
    'PRICE': resolve_file(['Data/kospi.csv', 'kospi.csv'], 'PRICE'),

    'EPS_FY2': resolve_file(
        ['Data/kospi_EPS_FY2.csv','kospi_EPS_FY2.csv',
         'Data/kospi_eps_fy2.csv','kospi_eps_fy2.csv'],
        'FY2 EPS REVISION'
    ),

    'SURPRISE': resolve_file(
        ['surprise.csv','Data/surprise.csv'],
        'DAILY EARNINGS SURPRISE'
    ),

    'MCAP': resolve_file(
        ['Data/kospi_stock.csv','kospi_stock.csv',
         'Data/kospi_mcap.csv','kospi_mcap.csv'],
        'MARKET CAP'
    ),

    # 파일명이 *_60이면 이미 60일 평균 거래대금 파일로 간주하고,
    # 일반 trading_value 파일이면 코드에서 60거래일 rolling mean을 계산한다.
    'TRADING_VALUE': resolve_file(
        ['Data/trading_value_60_0906.csv','trading_value_60_0906.csv',
         'Data/trading_value_60.csv','trading_value_60.csv',
         'Data/kospi_trading_value.csv','kospi_trading_value.csv',
         'Data/trading_value.csv','trading_value.csv'],
        'TRADING VALUE'
    ),

    'BENCHMARK': resolve_file(
        ['Data/kospi_200.csv','kospi_200.csv'],
        'KOSPI200 INDEX'
    ),

    'FACTOR': resolve_file(
        ['Data/factor_model_index.csv','factor_model_index.csv'],
        'FACTOR MODEL'
    )
}


## 2. 공통 숫자 변환 함수


### 코드 설명 — 숫자형 변환과 날짜 정렬: 문자열로 저장된 수치, 중복 날짜를 처리한다.


In [ ]:
def numeric_panel(frame):
    def convert(column):
        if pd.api.types.is_numeric_dtype(column.dtype):
            return column.astype(float)
        text = (
            column.astype('string')
            .str.strip()
            .str.replace(',', '', regex=False)
            .str.replace(r'^\((.+)\)$', r'-\1', regex=True)
        )
        return pd.to_numeric(text, errors='coerce').astype(float)
    return frame.apply(convert).replace([np.inf, -np.inf], np.nan)


def read_panel(path):
    try:
        df = pd.read_csv(path, index_col='Date', parse_dates=True, low_memory=False)
    except Exception:
        df = pd.read_csv(path, index_col=0, parse_dates=True, low_memory=False)

    df = numeric_panel(df)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    if not df.index.is_unique:
        df = df.groupby(level=0).last()
    return df


def latest_on_or_before(frame, date):
    idx = frame.index[frame.index <= date]
    if len(idx) == 0:
        return pd.Series(index=frame.columns, dtype=float)
    return frame.loc[idx[-1]]


## 3. 원자료 로드


### 코드 설명 — 실적·전망·시장 데이터 로딩: 서로 다른 빈도의 패널을 날짜 기준으로 정렬한다.


In [ ]:
price_df = read_panel(FILES['PRICE'])
fy2_revision_raw = read_panel(FILES['EPS_FY2'])
mcap_raw = read_panel(FILES['MCAP'])
trading_value_raw = read_panel(FILES['TRADING_VALUE'])

# Daily Earnings Surprise: 첫 열이 실제 날짜라고 가정
surprise_daily = pd.read_csv(FILES['SURPRISE'], low_memory=False)
date_col = surprise_daily.columns[0]
surprise_daily[date_col] = pd.to_datetime(surprise_daily[date_col], errors='coerce')
surprise_daily = (
    surprise_daily
    .dropna(subset=[date_col])
    .set_index(date_col)
    .sort_index()
)
surprise_daily = numeric_panel(surprise_daily)
if not surprise_daily.index.is_unique:
    surprise_daily = surprise_daily.groupby(level=0).last()

# EPS와 시가총액 자료의 날짜를 해당 월말로 정렬
for name in ['fy2_revision_raw', 'mcap_raw']:
    frame = globals()[name].copy()
    frame.index = pd.to_datetime(frame.index) + pd.offsets.MonthEnd(0)
    frame = frame.sort_index()
    if not frame.index.is_unique:
        frame = frame.groupby(level=0).last()
    globals()[name] = frame

print('Price:', price_df.index.min(), '~', price_df.index.max())
print('Surprise:', surprise_daily.index.min(), '~', surprise_daily.index.max())
print('FY2:', fy2_revision_raw.index.min(), '~', fy2_revision_raw.index.max())
print('Trading Value:', trading_value_raw.index.min(), '~', trading_value_raw.index.max())


## 4. 공통 종목 + 시가총액 + 60일 거래대금

`TradingValue60`은 종목별 **60거래일 평균 거래대금**이다.

일간 거래대금 원자료라면

$$
TV60_{i,t}=\frac{1}{60}\sum_{d=t-59}^{t}TradingValue_{i,d}
$$

을 계산한다. 파일명에 `_60`이 포함된 경우 이미 60일 지표라고 보고 그대로 월말 값으로 사용한다.

매월 시가총액 3,000억원 이상 종목 중 TradingValue60의 횡단면 percentile rank가 **20% 이하인 종목을 제거**한다.


### 코드 설명 — 종목 열 교집합과 거래대금 처리: 이미 계산된 60일 평균과 원시 일간 거래대금을 구분한다.


In [ ]:
common_cols = (
    price_df.columns
    .intersection(fy2_revision_raw.columns)
    .intersection(surprise_daily.columns)
    .intersection(mcap_raw.columns)
    .intersection(trading_value_raw.columns)
)

price_df = price_df.reindex(columns=common_cols)
fy2_revision_raw = fy2_revision_raw.reindex(columns=common_cols)
surprise_daily = surprise_daily.reindex(columns=common_cols)
mcap_raw = mcap_raw.reindex(columns=common_cols)
trading_value_raw = trading_value_raw.reindex(columns=common_cols)

# 시가총액: 각 월 마지막 관측값
mcap_monthly = mcap_raw.resample('ME').last()

# 60일 거래대금 파일인지 파일명으로 우선 판별
trading_value_path = str(FILES['TRADING_VALUE']).lower()
if '60' in Path(trading_value_path).stem:
    trading_value_60_monthly = trading_value_raw.resample('ME').last()
    print('Trading Value: precomputed 60-day series detected')
else:
    # 일반 거래대금은 일간 자료를 전제로 60거래일 평균 계산
    trading_value_60_daily = (
        trading_value_raw
        .rolling(TRADING_VALUE_LOOKBACK, min_periods=TRADING_VALUE_LOOKBACK)
        .mean()
    )
    trading_value_60_monthly = trading_value_60_daily.resample('ME').last()
    print('Trading Value: 60-trading-day rolling mean calculated from raw series')

print('Common usable stocks:', len(common_cols))


## 5. Earnings Surprise + FY2 EPS Revision Score + 3개월 Lag

본 버전에서는 **FY1을 사용하지 않고 FY2 EPS Revision만 사용**한다.

$$
RevisionScore_t = FY2Revision_t
$$

그리고 EPS Revision과 Earnings Surprise 모두 formation month 기준 3개월 전 값을 사용한다.

$$
RevisionSignal_t=RevisionScore_{t-3}
$$

$$
SurpriseSignal_t=Surprise_{t-3}
$$

이후 formation month $t$에서 종목을 고르고 실제 수익률은 다음 달 $t+1$에 적용한다.


### 코드 설명 — 정보 이용 가능 시점: Surprise를 월말로 변환하고 FY2와 함께 3개월 lag를 적용한다. 실제 공시일을 확인한 것은 아니다.


In [ ]:
# Daily Surprise -> 각 월 마지막 관측값
surprise_monthly = surprise_daily.resample('ME').last()
surprise_signal_raw = surprise_monthly / 100.0

month_idx = pd.date_range(
    start=fy2_revision_raw.index.min(),
    end=fy2_revision_raw.index.max(),
    freq='ME'
)

fy2_revision = fy2_revision_raw.reindex(month_idx) / 100.0

# FY2 only
revision_score_raw = fy2_revision.copy()

# 핵심 변경: 3개월 availability lag
revision_score = revision_score_raw.shift(SIGNAL_LAG_MONTHS)
surprise_signal = surprise_signal_raw.shift(SIGNAL_LAG_MONTHS)

print(f'Revision / Surprise signals shifted by {SIGNAL_LAG_MONTHS} months')
display(revision_score.iloc[-5:, :8])
display(surprise_signal.iloc[-5:, :8])


## 6. 월별 수익률 / 백테스트 기간 / 공매도 금지월


### 코드 설명 — 형성 월과 보유 월 분리: 공매도 금지기간과 완전히 종료된 월을 확인한다.


In [ ]:
price_monthly = price_df.resample('ME').last()
monthly_return = price_monthly.pct_change(fill_method=None)

# 실행일 기준 완전히 종료된 최근 월까지만 사용
today = pd.Timestamp.today().normalize()
latest_completed_month = today - pd.offsets.MonthEnd(1)

data_last_months = [
    price_monthly.dropna(how='all').index.max(),
    revision_score.dropna(how='all').index.max(),
    surprise_signal.dropna(how='all').index.max(),
    mcap_monthly.dropna(how='all').index.max(),
    trading_value_60_monthly.dropna(how='all').index.max(),
]
latest_data_month = min(pd.Timestamp(x) for x in data_last_months if pd.notna(x))
BACKTEST_END = min(latest_completed_month, latest_data_month)

all_holding_dates = pd.date_range(BACKTEST_START, BACKTEST_END, freq='ME')


def month_overlaps_ban(month_end):
    p = pd.Timestamp(month_end).to_period('M')
    month_start = p.start_time.normalize()
    month_last = p.end_time.normalize()
    return any(month_start <= ban_end and month_last >= ban_start
               for ban_start, ban_end in SHORT_BAN_RANGES)

ban_month_mask = pd.Series(
    [month_overlaps_ban(d) for d in all_holding_dates],
    index=all_holding_dates,
    name='Short Sale Ban Month'
)

# formation은 항상 holding month 직전 월말
formation_dates = all_holding_dates - pd.offsets.MonthEnd(1)
holding_dates = all_holding_dates

print('Backtest:', BACKTEST_START.date(), '~', BACKTEST_END.date())
print('Excluded full-ban months:', int(ban_month_mask.sum()))
print('Active months:', int((~ban_month_mask).sum()))
display(pd.DataFrame({'Ban Month': ban_month_mask[ban_month_mask].index}))


## 7. Liquidity Filter + Combined Score + Long/Short 선정

각 formation month에서 다음 순서로 종목을 선별한다.

1. 시가총액 3,000억원 이상
2. EPS Revision, Earnings Surprise, TradingValue60 모두 존재
3. TradingValue60 횡단면 하위 20% 제외
4. 남은 종목 안에서 Revision Rank와 Surprise Rank 계산
5. `Combined Score = 0.5 × Revision Rank + 0.5 × Surprise Rank`
6. 상위 20개 Long, 하위 20개 Short

정상 운용월의 목표 익스포저는

$$
Long=+100\%,\qquad Short=-100\%
$$

$$
Net=0\%,\qquad Gross=200\%
$$


### 코드 설명 — 결측 제거와 유니버스 선정: 시총, 유동성, 신호 순위를 당시 자료에서 계산한다.


In [ ]:
long_selections = {}
short_selections = {}
signal_snapshots = {}
coverage_rows = []

for formation_date, holding_date in zip(formation_dates, holding_dates):

    # 전면 공매도 금지와 겹치는 holding month는 전략 미운용
    if ban_month_mask.loc[holding_date]:
        continue

    revision = latest_on_or_before(revision_score, formation_date).reindex(common_cols)
    surprise = latest_on_or_before(surprise_signal, formation_date).reindex(common_cols)
    mcap = latest_on_or_before(mcap_monthly, formation_date).reindex(common_cols)
    tv60 = latest_on_or_before(trading_value_60_monthly, formation_date).reindex(common_cols)

    x = pd.DataFrame({
        'Revision': revision,
        'Surprise': surprise,
        'MCap': mcap,
        'TradingValue60': tv60,
    })

    # 1) 시총 필터
    x = x[x['MCap'] >= MCAP_MIN_EOK].copy()

    # 2) 핵심 데이터 결측 제거
    x = x.dropna(subset=['Revision', 'Surprise', 'TradingValue60'])

    before_liquidity_n = len(x)

    # 3) 거래대금 하위 20% 제외
    x['TradingValue Rank'] = x['TradingValue60'].rank(pct=True, method='average')
    x = x[x['TradingValue Rank'] > LIQUIDITY_BOTTOM_EXCLUDE].copy()

    # 4) 신호 Rank는 최종 tradable universe 안에서 다시 계산
    x['Revision Rank'] = x['Revision'].rank(pct=True, method='average')
    x['Surprise Rank'] = x['Surprise'].rank(pct=True, method='average')

    # 5) Combined Score
    x['Combined Score'] = (
        REVISION_WEIGHT * x['Revision Rank']
        + SURPRISE_WEIGHT * x['Surprise Rank']
    )

    signal_snapshots[formation_date] = x.copy()

    coverage_rows.append({
        'Formation Date': formation_date,
        'Holding Date': holding_date,
        'Before Liquidity Filter N': before_liquidity_n,
        'After Liquidity Filter N': len(x),
        'Liquidity Excluded %': (1 - len(x) / before_liquidity_n) if before_liquidity_n else np.nan,
    })

    if len(x) >= (N_LONG + N_SHORT):
        long_selections[formation_date] = (
            x.sort_index()
             .sort_values('Combined Score', ascending=False, kind='stable')
             .head(N_LONG).index.tolist()
        )
        short_selections[formation_date] = (
            x.sort_index()
             .sort_values('Combined Score', ascending=True, kind='stable')
             .head(N_SHORT).index.tolist()
        )

coverage = pd.DataFrame(coverage_rows)
if len(coverage):
    coverage = coverage.set_index('Formation Date')
    display(coverage.tail(12))

print('Valid long formation months:', len(long_selections))
print('Valid short formation months:', len(short_selections))


## 8. +100% Long / -100% Short 가중치


### 코드 설명 — 포트폴리오 가중치: 롱·숏 종목별 명목 비중을 구성한다.


In [ ]:
def make_long_weights(selections):
    weights = pd.DataFrame(0.0, index=price_monthly.index, columns=common_cols)
    for date, names in selections.items():
        if date in weights.index and len(names):
            weights.loc[date, names] = 1.0 / len(names)
    return weights


def make_short_weights(selections):
    weights = pd.DataFrame(0.0, index=price_monthly.index, columns=common_cols)
    for date, names in selections.items():
        if date in weights.index and len(names):
            weights.loc[date, names] = -1.0 / len(names)
    return weights

long_target = make_long_weights(long_selections)
short_target = make_short_weights(short_selections)
long_short_target = long_target + short_target

active = long_short_target.abs().sum(axis=1) > 0
exposure_check = pd.DataFrame({
    'Long Exposure': long_target.sum(axis=1),
    'Short Exposure': short_target.sum(axis=1),
    'Net Exposure': long_short_target.sum(axis=1),
    'Gross Exposure': long_short_target.abs().sum(axis=1),
})

print('Median exposures in active formation months')
display(exposure_check.loc[active].median().to_frame('Median'))


## 9. 다음 달 보유 + 거래비용 + 연 3% 대차비용

Formation month $t$에 정한 가중치는 다음 달 $t+1$ 수익률에 적용한다.

공매도 비용은 Short 명목금액에 대해 연 3%를 월 복리 기준으로 환산하여 매월 차감한다.

$$
BorrowCost_t = ShortNotional_t\times\left((1+0.03)^{1/12}-1\right)
$$

공매도 금지월에는 Long/Short 목표비중을 모두 0으로 만들어 전략을 쉬도록 한다. 금지 시작 시 기존 포지션 청산 비용, 재개 시 신규 진입 비용도 반영된다.


### 코드 설명 — 비용 반영: 다음 달 수익률과 포지션 변동, 대차 비용을 계산한다.


In [ ]:
def formation_to_holding_weights(formation_weights):
    # formation t -> holding t+1
    return formation_weights.shift(1).reindex(holding_dates).fillna(0.0)


def force_zero_during_ban(holding_weights):
    w = holding_weights.copy()
    banned = ban_month_mask.reindex(w.index).fillna(False)
    w.loc[banned.values, :] = 0.0
    return w


def drift_weights_by_sleeve(target, asset_ret):
    # Long/Short 각 sleeve의 월말 드리프트를 별도로 계산
    long_w = target.clip(lower=0)
    short_w = target.clip(upper=0)

    long_end = long_w * (1 + asset_ret)
    if long_end.sum() > 0:
        long_end = long_end / long_end.sum()

    short_abs_end = (-short_w) * (1 + asset_ret)
    if short_abs_end.sum() > 0:
        short_end = -(short_abs_end / short_abs_end.sum())
    else:
        short_end = short_abs_end * 0.0

    return long_end + short_end


def backtest_with_long_short_costs(
    target_weights,
    returns,
    one_way_cost=ONE_WAY_COST,
    monthly_borrow_cost=MONTHLY_BORROW_COST,
    liquidate_at_end=True
):
    target_weights = target_weights.reindex(holding_dates).fillna(0.0)
    returns = returns.reindex(holding_dates)

    prev_end_weights = pd.Series(0.0, index=target_weights.columns)
    records = []

    for date in target_weights.index:
        target = target_weights.loc[date].astype(float)
        asset_ret = returns.loc[date].reindex(target.index)

        active = target.ne(0)
        if active.any() and asset_ret[active].isna().any():
            missing = asset_ret[active][asset_ret[active].isna()].index.tolist()
            raise ValueError(f'Missing held-stock returns at {date.date()}: {missing[:10]}')

        asset_ret = asset_ret.fillna(0.0)

        # Long 매매 + Short 진입/환매 모두 절대 포지션 변화에 편도 비용 적용
        traded = target - prev_end_weights
        gross_turnover = traded.abs().sum()
        trading_cost = one_way_cost * gross_turnover

        short_notional = float((-target.clip(upper=0)).sum())
        borrow_cost = monthly_borrow_cost * short_notional

        gross_return = float((target * asset_ret).sum())
        net_return = gross_return - trading_cost - borrow_cost

        if target.abs().sum() > 0:
            prev_end_weights = drift_weights_by_sleeve(target, asset_ret)
        else:
            # 공매도 금지월 등 미운용월에는 현금 상태
            prev_end_weights = pd.Series(0.0, index=target.index)

        records.append({
            'Date': date,
            'Strategy Active': bool(target.abs().sum() > 0),
            'Long Exposure': float(target.clip(lower=0).sum()),
            'Short Exposure': float(target.clip(upper=0).sum()),
            'Gross Exposure': float(target.abs().sum()),
            'Net Exposure': float(target.sum()),
            'Gross Return': gross_return,
            'Gross Turnover': float(gross_turnover),
            'Transaction Cost': float(trading_cost),
            'Short Notional': short_notional,
            'Borrow Cost': float(borrow_cost),
            'Net Return': float(net_return),
        })

    detail = pd.DataFrame(records).set_index('Date')
    detail['Liquidation Cost'] = 0.0

    if liquidate_at_end and len(detail):
        final_turnover = float(prev_end_weights.abs().sum())
        liquidation_cost = one_way_cost * final_turnover
        detail.loc[detail.index[-1], 'Transaction Cost'] += liquidation_cost
        detail.loc[detail.index[-1], 'Liquidation Cost'] = liquidation_cost
        detail.loc[detail.index[-1], 'Net Return'] -= liquidation_cost

    return detail['Net Return'], detail

holding_long = force_zero_during_ban(formation_to_holding_weights(long_target))
holding_short = force_zero_during_ban(formation_to_holding_weights(short_target))
holding_long_short = force_zero_during_ban(formation_to_holding_weights(long_short_target))

holding_weight_sets = {
    'Long +100%': holding_long,
    'Short -100%': holding_short,
    'Long-Short 200% Gross': holding_long_short,
}

backtest_details = {}
net_returns = {}

for label, held_weights in holding_weight_sets.items():
    r, detail = backtest_with_long_short_costs(
        held_weights,
        monthly_return,
        one_way_cost=ONE_WAY_COST,
        monthly_borrow_cost=MONTHLY_BORROW_COST,
        liquidate_at_end=LIQUIDATE_AT_END,
    )
    net_returns[label] = r
    backtest_details[label] = detail

result = pd.DataFrame(net_returns).reindex(holding_dates)

print('Main strategy exposure check')
display(backtest_details['Long-Short 200% Gross'][
    ['Strategy Active','Long Exposure','Short Exposure','Net Exposure','Gross Exposure']
].tail(24))

cost_summary = pd.DataFrame({
    label: {
        'Avg Monthly Gross Turnover': d['Gross Turnover'].mean(),
        'Total Transaction Cost': d['Transaction Cost'].sum(),
        'Avg Short Notional': d['Short Notional'].mean(),
        'Total Borrow Cost': d['Borrow Cost'].sum(),
    }
    for label, d in backtest_details.items()
}).T

display(cost_summary)


## 10. 성과지표


### 코드 설명 — 성과 측정: 전략과 벤치마크 수익률을 비교한다.


In [ ]:
benchmark = pd.read_csv(
    FILES['BENCHMARK'], index_col=0, parse_dates=True, low_memory=False
).sort_index()
benchmark_level = pd.to_numeric(benchmark.iloc[:, 0], errors='coerce').where(lambda x: x > 0)
benchmark_return = (
    benchmark_level.resample('ME').last().pct_change(fill_method=None)
    .reindex(holding_dates).rename('KOSPI200')
)


def metrics(r):
    r = r.dropna()
    if len(r) == 0:
        return (np.nan,) * 5
    nav = (1 + r).cumprod()
    total = nav.iloc[-1] - 1
    cagr = nav.iloc[-1] ** (12 / len(r)) - 1
    vol = r.std() * np.sqrt(12)
    sharpe = r.mean() * 12 / vol if vol > 0 else np.nan
    mdd = (nav / nav.cummax() - 1).min()
    return total, cagr, vol, sharpe, mdd

comparison = pd.concat([result, benchmark_return], axis=1)
performance = pd.DataFrame(
    {c: metrics(comparison[c]) for c in comparison.columns},
    index=['Total Return','CAGR','Annual Volatility','Sharpe Ratio','MDD']
)

print('Performance — net of transaction costs + 3% annual borrow cost')
display(performance)


## 11. CAPM / FF3 회귀분석 — Net Return


### 코드 설명 — CAPM 및 FF3: 월별 요인 지수 변화율과 Rf를 정렬하고 OLS 및 HAC(12)로 알파, t값, p값을 계산한다.


In [ ]:
factor = read_panel(FILES['FACTOR'])
required_factors = ['KOSPI', 'SMB', 'HML', 'Rf']
missing_columns = set(required_factors).difference(factor.columns)
if missing_columns:
    raise ValueError(f'Missing factor columns: {sorted(missing_columns)}')

f = factor[required_factors].resample('ME').last()
monthly_factors = f[['KOSPI','SMB','HML']].pct_change(fill_method=None)
rf = ((1 + f['Rf'] / 100).where(f['Rf'] > -100) ** (1 / 12) - 1)
monthly_factors = monthly_factors.rename(columns={'KOSPI':'MKT_RAW'})
monthly_factors['RF'] = rf
monthly_factors['MKT'] = monthly_factors['MKT_RAW'] - monthly_factors['RF']

REGRESSION_HAC_LAGS = 12
summary_rows = []
regression_models = {}

for label in result.columns:
    sample = pd.concat(
        [result[label].rename('Portfolio Return'), monthly_factors[['RF','MKT','SMB','HML']]],
        axis=1
    ).reindex(holding_dates).replace([np.inf,-np.inf], np.nan).dropna()

    sample['Excess Return'] = sample['Portfolio Return'] - sample['RF']

    for model_name, factors_used in [('CAPM',['MKT']), ('FF3',['MKT','SMB','HML'])]:
        X = sm.add_constant(sample[factors_used], has_constant='add')
        fit = sm.OLS(sample['Excess Return'], X).fit(
            cov_type='HAC', cov_kwds={'maxlags': REGRESSION_HAC_LAGS}
        )
        regression_models[(label, model_name)] = fit
        summary_rows.append({
            'Portfolio': label,
            'Model': model_name,
            'Annualized Alpha': fit.params['const'] * 12,
            'Alpha t-stat': fit.tvalues['const'],
            'Alpha p-value': fit.pvalues['const'],
            'Market Beta': fit.params.get('MKT', np.nan),
            'SMB Beta': fit.params.get('SMB', np.nan),
            'HML Beta': fit.params.get('HML', np.nan),
            'R2': fit.rsquared,
            'N': int(fit.nobs),
        })

regression_summary = pd.DataFrame(summary_rows).set_index(['Portfolio','Model'])
display(regression_summary)
print('\nMain Long-Short strategy')
display(regression_summary.loc['Long-Short 200% Gross'])


## 12. Turnover / 연도별 수익률 / NAV


### 코드 설명 — 회전율 및 NAV: 연도별 성과와 누적 성과를 정리한다.


In [ ]:
turn = pd.DataFrame({
    label: detail['Gross Turnover'] / 2
    for label, detail in backtest_details.items()
})

display(pd.DataFrame({
    'Average Monthly One-way Turnover': turn.mean(),
    'Median Monthly One-way Turnover': turn.median(),
}))

annual = {}
for c in result.columns:
    r = result[c].dropna()
    annual[c] = (1 + r).groupby(r.index.year).prod() - 1

display(pd.DataFrame(annual))

nav = pd.concat([result, benchmark_return], axis=1)
(1 + nav.fillna(0)).cumprod().plot(
    figsize=(12,6),
    title='EPS Revision + Earnings Surprise Long-Short — 3M Lag, Liquidity Filter, Net Costs'
)
plt.ylabel('NAV')
plt.grid()
plt.show()


## 13. 최근 Long / Short 보유종목


### 코드 설명 — 선정 기록 확인: 종목과 신호 점수를 검토한다.


In [ ]:
rows = []
for date, names in long_selections.items():
    x = signal_snapshots[date]
    for name in names:
        rows.append({
            'Formation Date': date,
            'Side': 'LONG',
            'Stock': name,
            'MCap': x.at[name,'MCap'],
            'TradingValue60': x.at[name,'TradingValue60'],
            'TradingValue Rank': x.at[name,'TradingValue Rank'],
            'Revision': x.at[name,'Revision'],
            'Revision Rank': x.at[name,'Revision Rank'],
            'Surprise': x.at[name,'Surprise'],
            'Surprise Rank': x.at[name,'Surprise Rank'],
            'Combined Score': x.at[name,'Combined Score'],
            'Weight': 1 / N_LONG,
        })
for date, names in short_selections.items():
    x = signal_snapshots[date]
    for name in names:
        rows.append({
            'Formation Date': date,
            'Side': 'SHORT',
            'Stock': name,
            'MCap': x.at[name,'MCap'],
            'TradingValue60': x.at[name,'TradingValue60'],
            'TradingValue Rank': x.at[name,'TradingValue Rank'],
            'Revision': x.at[name,'Revision'],
            'Revision Rank': x.at[name,'Revision Rank'],
            'Surprise': x.at[name,'Surprise'],
            'Surprise Rank': x.at[name,'Surprise Rank'],
            'Combined Score': x.at[name,'Combined Score'],
            'Weight': -1 / N_SHORT,
        })

holdings = pd.DataFrame(rows)
if len(holdings):
    latest_date = holdings['Formation Date'].max()
    print('Latest active formation:', latest_date.date())
    display(
        holdings[holdings['Formation Date'] == latest_date]
        .sort_values(['Side','Combined Score'], ascending=[True,False])
    )


## 14. 최신 Signal 기준 Long / Short 포트폴리오

최신 추천에도 동일하게 **3개월 lag**와 **거래대금 하위 20% 제외**를 적용한다.


### 코드 설명 — 최신 신호: 동일한 시차 및 유동성 조건으로 후보를 계산한다.


In [ ]:
latest_candidate_date = min(
    revision_score.dropna(how='all').index.max(),
    surprise_signal.dropna(how='all').index.max(),
    mcap_monthly.dropna(how='all').index.max(),
    trading_value_60_monthly.dropna(how='all').index.max(),
)
LATEST_SIGNAL_DATE = pd.Timestamp(latest_candidate_date)

latest_signal = pd.DataFrame({
    'Revision': latest_on_or_before(revision_score, LATEST_SIGNAL_DATE).reindex(common_cols),
    'Surprise': latest_on_or_before(surprise_signal, LATEST_SIGNAL_DATE).reindex(common_cols),
    'MCap': latest_on_or_before(mcap_monthly, LATEST_SIGNAL_DATE).reindex(common_cols),
    'TradingValue60': latest_on_or_before(trading_value_60_monthly, LATEST_SIGNAL_DATE).reindex(common_cols),
})

latest_signal = latest_signal[
    latest_signal['MCap'] >= MCAP_MIN_EOK
].dropna(subset=['Revision','Surprise','TradingValue60'])

latest_signal['TradingValue Rank'] = latest_signal['TradingValue60'].rank(pct=True, method='average')
latest_signal = latest_signal[
    latest_signal['TradingValue Rank'] > LIQUIDITY_BOTTOM_EXCLUDE
].copy()

latest_signal['Revision Rank'] = latest_signal['Revision'].rank(pct=True, method='average')
latest_signal['Surprise Rank'] = latest_signal['Surprise'].rank(pct=True, method='average')
latest_signal['Combined Score'] = (
    REVISION_WEIGHT * latest_signal['Revision Rank']
    + SURPRISE_WEIGHT * latest_signal['Surprise Rank']
)

latest_long = latest_signal.sort_values('Combined Score', ascending=False).head(N_LONG).copy()
latest_short = latest_signal.sort_values('Combined Score', ascending=True).head(N_SHORT).copy()
latest_long['Portfolio Weight'] = 1 / len(latest_long)
latest_short['Portfolio Weight'] = -1 / len(latest_short)

cols = [
    'MCap','TradingValue60','TradingValue Rank',
    'Revision','Revision Rank','Surprise','Surprise Rank',
    'Combined Score','Portfolio Weight'
]

print(f'Latest signal date: {LATEST_SIGNAL_DATE.date()} | Universe N = {len(latest_signal)}')
print('\nLATEST LONG (+100%)')
display(latest_long[cols])
print('\nLATEST SHORT (-100%)')
display(latest_short[cols])


# 최종 전략 명세

| 항목 | 설정 |
|---|---|
| 유니버스 | 매월 시가총액 3,000억원 이상 |
| EPS Revision | **FY2만 사용** |
| Earnings Surprise | 월별 마지막 관측값 |
| Signal Lag | 3개월 |
| Liquidity | 60거래일 평균 거래대금 하위 20% 제외 |
| Score | Revision Rank 50% + Surprise Rank 50% |
| Long | 상위 20개, 총 +100% |
| Short | 하위 20개, 총 -100% |
| Net Exposure | 0% |
| Gross Exposure | 200% |
| 리밸런싱 | 월 1회 |
| 수익률 적용 | formation 다음 달 |
| 거래비용 | 편도 15bp / 왕복 30bp |
| 대차비용 | 연 3% |
| 공매도 전면 금지 | 겹치는 월 전략 미운용(0% exposure) |
| 회귀 | CAPM / FF3, HAC(12) |

중요한 시점 구조는 다음과 같다.

$$
Signal_{t-3}
\rightarrow
Formation_t
\rightarrow
Holding_{t+1}
$$

즉 **3개월 신호 lag와 1개월 formation-to-holding lag는 서로 별개**로 적용된다.

> 주의: 2021년 5월 이후 2023년 11월 전면 금지 재개 전까지는 한국 시장에서 공매도가 KOSPI200·KOSDAQ150 종목 중심으로 부분 재개된 기간이 있었다. 이 노트북은 전면 금지기간만 제외하므로, 그 부분재개 기간의 종목별 실제 shortability까지 완전히 재현하려면 당시 역사적 구성종목 또는 종목별 대차 가능 데이터를 추가하는 것이 더 엄밀하다.


# NAV 비교 및 CSV 저장

아래에서는 먼저 Long과 Short sleeve의 NAV를 한 그래프에 표시하고,
그 다음 메인 Long-Short 전략과 KOSPI200 Benchmark만 따로 비교한다.

마지막으로 두 번째 그래프에 사용한 **Long-Short NAV와 KOSPI200 NAV**를
`earning_nav.csv`로 저장한다.


### 코드 설명 — 그래프 및 CSV 저장: 누적 성과를 시각화한다.


In [ ]:
# ============================================
# 15. NAV 비교 그래프 + earning_nav.csv 저장
# ============================================

# 1) Long / Short sleeve NAV
long_short_sleeve_nav = pd.DataFrame({
    'Long NAV': (1 + result['Long +100%'].fillna(0)).cumprod(),
    'Short NAV': (1 + result['Short -100%'].fillna(0)).cumprod(),
})

ax = long_short_sleeve_nav.plot(
    figsize=(12, 6),
    title='Long vs Short Sleeve NAV'
)
ax.set_ylabel('NAV')
ax.set_xlabel('Date')
ax.grid(True)
plt.show()


# 2) 메인 Long-Short 전략 vs KOSPI200 Benchmark
main_vs_benchmark = pd.DataFrame({
    'Long-Short NAV': (1 + result['Long-Short 200% Gross'].fillna(0)).cumprod(),
    'KOSPI200 NAV': (1 + benchmark_return.fillna(0)).cumprod(),
})

ax = main_vs_benchmark.plot(
    figsize=(12, 6),
    title='Long-Short Strategy vs KOSPI200'
)
ax.set_ylabel('NAV')
ax.set_xlabel('Date')
ax.grid(True)
plt.show()


# 3) 두 NAV를 CSV로 저장
earning_nav = main_vs_benchmark.copy()
earning_nav.index.name = 'Date'

CSV_OUTPUT = 'earning_nav.csv'
earning_nav.to_csv(CSV_OUTPUT, encoding='utf-8-sig')

print(f'Saved: {CSV_OUTPUT}')
display(earning_nav.tail())
